[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Saving a Model &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import json
import shutil
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]
SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)

print(f"scikit-learn {sklearn.__version__} | joblib {joblib.__version__} | {len(network)} stations")

scikit-learn 1.6.1 | joblib 1.6.0 | 1200 stations


**1.** Fit `RandomForestRegressor(n_estimators=50, random_state=0)` on `X_train` and `y_train`, save it to `scratch/forest.joblib`, load it back, and check that its predictions for `X_test` are the same as the original's.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    network[FEATURES], network["jan_mean"], test_size=0.2, random_state=0
)
forest = RandomForestRegressor(n_estimators=50, random_state=0).fit(X_train, y_train)
joblib.dump(forest, SCRATCH / "forest.joblib")
again = joblib.load(SCRATCH / "forest.joblib")

print(np.array_equal(again.predict(X_test), forest.predict(X_test)))

True


The split is made again with the worked examples' seed, since this notebook runs on its own. A forest saves all fifty of its trees, and the loaded copy predicts exactly what the original does.

**2.** Save `model` again with `joblib.dump(model, SCRATCH / "small.joblib", compress=3)`, load that file, and check that its predictions match the uncompressed one's.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    network[FEATURES], network["jan_mean"], test_size=0.2, random_state=0
)
model = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
joblib.dump(model, SCRATCH / "jan_model.joblib")
joblib.dump(model, SCRATCH / "small.joblib", compress=3)

plain = joblib.load(SCRATCH / "jan_model.joblib")
small = joblib.load(SCRATCH / "small.joblib")
print(np.array_equal(plain.predict(X_test), small.predict(X_test)))
sizes = {name: (SCRATCH / name).stat().st_size for name in ["small.joblib", "jan_model.joblib"]}
print("smaller:", sizes["small.joblib"] < sizes["jan_model.joblib"])

True
smaller: True


`compress` makes the file smaller, and loading needs nothing extra, because joblib notices the compression itself. The cell compares the sizes instead of printing them, since a file's size depends on the versions that wrote it.

**3.** Read `scratch/jan_model.json` back with `json.loads`, and print the scikit-learn version it records beside `sklearn.__version__`, with whether they are equal.

In [4]:
about = {"model": "jan_model.joblib", "versions": {"scikit-learn": sklearn.__version__}}
(SCRATCH / "jan_model.json").write_text(json.dumps(about, indent=2))

recorded = json.loads((SCRATCH / "jan_model.json").read_text())["versions"]["scikit-learn"]
print(recorded, sklearn.__version__, recorded == sklearn.__version__)

1.6.1 1.6.1 True


The first two lines write a small version of the worked examples' file, so this answer runs on its own. A program that loads a model can make this check first and stop with a clear message.

**4.** Make loading a file from another scikit-learn version fail instead of warn. Save a forest under the version `1.5.2`, as Common errors does, then load it inside `warnings.catch_warnings()` after calling `warnings.simplefilter("error", InconsistentVersionWarning)`. The warning is raised as an exception of its own class, so catch it with `except InconsistentVersionWarning as error`.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    network[FEATURES], network["jan_mean"], test_size=0.2, random_state=0
)
forest = RandomForestRegressor(n_estimators=50, random_state=0).fit(X_train, y_train)

running = sklearn.base.__version__
sklearn.base.__version__ = "1.5.2"
joblib.dump(forest, SCRATCH / "forest_1_5_2.joblib")
sklearn.base.__version__ = running

with warnings.catch_warnings():
    warnings.simplefilter("error", InconsistentVersionWarning)
    try:
        joblib.load(SCRATCH / "forest_1_5_2.joblib")
    except InconsistentVersionWarning as error:
        print("stopped at:", error.estimator_name,
              error.original_sklearn_version, "->", error.current_sklearn_version)

stopped at: DecisionTreeRegressor 1.5.2 -> 1.6.1


With the filter set to `"error"`, the first warning is raised as an exception, so a program that loads models can refuse a mismatched file instead of carrying on. The exception names the class it stopped at, here a tree inside the forest, and both versions.

The answers are finished with their files, so this cell removes the scratch folder:

In [6]:
shutil.rmtree(SCRATCH)

print("scratch still there:", SCRATCH.exists())

scratch still there: False


---

&#8592; **Back to:** [Saving a Model](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/13-saving-a-model.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
